In [ ]:
import pandas as pd             # biblioteca de analise de dados
import re                       #expressão regulares
from datetime import datetime   # para trabalhar com datas

print("Pandas Versão:", pd.__version__)


Carregando base

In [ ]:
caminho_df = r"F:\Senai_dados\aulasDados\turma-visualizacao-de-dados\alunos\victor_h_santos\exercicios\semana_4\base_rh.csv" # r antes do caminho para não dar erro
df = pd.read_csv(caminho_df,
                 sep=";",           #Separador de colunas
                 encoding="cp1252",  #encoding do win para ler acentos, podendo ser utf8
                 decimal=","        #Separador decimal
                 )
print("-- Dados carregados com sucesso--")
print(f"O Dataset tem {df.shape[1]} colunas e {df.shape[0]} linhas")
print("\n  Colunas do dataset")
print(df.columns.tolist())




# analisando base

In [ ]:
df.head(5) #visualização das primeiras linhas
df.tail(5) #visualização das ultimas linhas
print("quantidade de nulos por coluna:")
print(df.isnull().sum())
print("percentual de nulos por coluna:")
print(df.isnull().sum() /df.count() *100)


In [ ]:
print("\n  Valores únicos por coluna:")
print(df.nunique())

Resumos

In [ ]:

df.info() # mostra os tipos e valores não nulos
df.describe().round()# describe traz os dados, round arredonda

In [ ]:

print('Quantidade de nulos por coluna:')
print(df.isnull().sum())

print()

# Percentual de nulos — mais útil em bases grandes
# len(df) = número total de linhas
print('Percentual de nulos por coluna:')
print((df.isnull().sum() / len(df) * 100).round(2))

---
## 🗂️ CÉLULA 6 — Explorando colunas categóricas

**Coluna categórica** = coluna com poucos valores possíveis (departamento, cargo, status...)  

`nunique()` conta quantos valores diferentes existem.  
`value_counts()` mostra a distribuição — quantas vezes cada valor aparece.

In [ ]:
# df.nunique() → retorna quantos valores ÚNICOS cada coluna tem
# Serve para identificar quais são categóricas (poucas opções)
# e quais são identificadoras (muitas opções diferentes)

print('Valores únicos por coluna:')
print(df.nunique())
# df['coluna'].value_counts() → conta quantas vezes cada valor aparece,
# ordenado do mais frequente para o menos frequente

# Vamos ver todas as colunas categóricas de uma vez usando um loop:
# 'for col in lista' = para cada coluna na lista, executa o bloco abaixo

for col in ['Departamento', 'Cargo', 'Genero', 'Status', 'Estado_Civil']:
    print(f'\n── {col} ──')   # f-string: {} insere o valor da variável
    print(df[col].value_counts())

---
## 🔎 CÉLULA 7 — Selecionando colunas e filtrando linhas

São as operações mais usadas no dia a dia. Você vai usar isso em todo projeto.

**Seleção de coluna** → `df['Nome']`  
**Seleção de múltiplas colunas** → `df[['Nome', 'Salario']]`  
**Filtro de linhas** → `df[df['Status'] == 'Ativo']`

In [ ]:
# FILTRO SIMPLES: selecionar só as linhas onde Status == 'Ativo'
# df['Status'] == 'Ativo' → cria uma Series de True/False
# df[ True/False ]        → mantém só as linhas onde é True

ativos = df[df['Status'] == 'Ativo']

print(f'Total de funcionários : {len(df)}')
print(f'Funcionários ativos   : {len(ativos)}')
print(f'Funcionários inativos : {len(df) - len(ativos)}')

In [ ]:
# FILTRO COMPOSTO: duas condições ao mesmo tempo
# Use & para AND (as duas condições precisam ser verdadeiras)
# Use | para OR  (pelo menos uma precisa ser verdadeira)
# IMPORTANTE: cada condição precisa estar entre parênteses

gerentes_ti = df[
    (df['Cargo'] == 'Gerente') & (df['Departamento'] == 'TI')
]

print(f'Gerentes na área de TI: {len(gerentes_ti)}')
print()
gerentes_ti[['Nome', 'Salario', 'Status', 'Data_Admissao']]

---
## 📁 CÉLULA 8 — Exportando para JSON

**O que é JSON?**  
É o formato padrão de troca de dados entre sistemas modernos — APIs, ERPs na nuvem, dashboards web.  
JSON = *JavaScript Object Notation*

```json
[
  { "Nome": "Julia Nunes", "Departamento": "Logística", "Salario": 9088.34 },
  { "Nome": "Gustavo Duarte", "Departamento": "TI", "Salario": 8155.98 }
]
```

**Por que `orient="records"`?**  
Define o formato do JSON. `records` = cada linha vira um objeto separado.  
É o formato que APIs REST esperam receber.

In [ ]:
import json  # biblioteca nativa do Python para manipular JSON

# df.to_json() → converte o DataFrame para JSON e salva no arquivo
df.to_json(
    'base_rh.json',
    orient='records',      # cada linha vira um objeto {}
    force_ascii=False,     # False = salva ã, é, ç como estão
    indent=2               # 2 espaços de indentação → arquivo legível
)

print('✓ base_rh.json gerado!')

# Ler de volta para confirmar
df_json = pd.read_json('base_rh.json')
print(f'  Linhas lidas do JSON: {len(df_json)}')

# Visualizar os 2 primeiros registros no formato JSON
with open('base_rh.json', 'r', encoding='utf-8') as f:
    amostra = json.load(f)
print('\nPrimeiros 2 registros no JSON:')
print(json.dumps(amostra[:2], ensure_ascii=False, indent=2))

---
## 📊 CÉLULA 9 — Exportando para Excel

Excel ainda é o formato mais pedido por gestores em ambientes industriais.  
O pandas permite criar um arquivo com **múltiplas abas** — uma por departamento.

**O que é `ExcelWriter`?**  
É um gerenciador que permite escrever várias abas no mesmo arquivo `.xlsx`.  
O `with ... as writer:` garante que o arquivo seja fechado corretamente ao final.

> ⚠️ Precisa do `openpyxl` instalado: `pip install openpyxl`

In [20]:
# Exportação simples: tudo em uma aba
df.to_excel('base_rh.xlsx', index=False, sheet_name='Funcionarios')
print('✓ base_rh.xlsx (aba única) gerado!')

# Exportação avançada: uma aba por departamento
with pd.ExcelWriter('base_rh_por_depto.xlsx', engine='openpyxl') as writer:

    # Aba 1: dataset completo
    df.to_excel(writer, sheet_name='Todos', index=False)
    print('  Aba Todos: todos os registros')

    # df['Departamento'].unique() → lista de departamentos sem repetição
    # sorted() → ordena em ordem alfabética
    for depto in sorted(df['Departamento'].unique()):

        # Filtrar só os funcionários desse departamento
        df_depto = df[df['Departamento'] == depto]

        # Limite do Excel: nomes de aba têm no máximo 31 caracteres
        nome_aba = depto[:31]

        df_depto.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f'  Aba "{nome_aba}": {len(df_depto)} registros')

print('\n✓ base_rh_por_depto.xlsx gerado!')

✓ base_rh.xlsx (aba única) gerado!
  Aba Todos: todos os registros
  Aba "Financeiro": 189 registros
  Aba "Logística": 156 registros
  Aba "Produção": 182 registros
  Aba "RH": 166 registros
  Aba "TI": 147 registros
  Aba "Vendas": 160 registros

✓ base_rh_por_depto.xlsx gerado!


---
## ☕ INTERVALO — 20h30

**Antes de continuar, confirme:**
- [ ] Os arquivos `base_rh.json` e `base_rh.xlsx` apareceram na sua pasta?
- [ ] O `df.shape` mostrou `(1000, 10)`?
- [ ] Você entendeu a diferença entre `head()` e `tail()`?

Se algum arquivo não apareceu, chame o professor agora — antes de continuar.

---
## 📅 CÉLULA 10 — Módulo datetime: o básico

**Por que isso importa?**  
A coluna `Data_Admissao` veio como **texto** (`object`).  
O Python não sabe que `'13/08/2024'` é uma data — pra ele é só uma string.  
Enquanto for texto, você **não consegue** calcular diferenças, filtrar por período, extrair mês ou ano.

**Dois comandos essenciais:**

| Comando | Significado | Exemplo |
|---|---|---|
| `strptime` | string → parse → time (lê um texto e vira data) | `'13/08/2024'` → `datetime(2024, 8, 13)` |
| `strftime` | string → format → time (pega uma data e formata como texto) | `datetime(2024, 8, 13)` → `'Agosto de 2024'` |

**Códigos de formato de data:**
| Código | Significado | Exemplo |
|---|---|---|
| `%d` | Dia com zero à esquerda | 08 |
| `%m` | Mês com zero à esquerda | 08 |
| `%Y` | Ano com 4 dígitos | 2024 |
| `%B` | Nome do mês por extenso | August |

In [25]:
from datetime import datetime, timedelta

# ── strptime: texto → objeto data ─────────────────────────────────────────
# Pegamos o primeiro valor da coluna Data_Admissao como exemplo
data_texto = '13/08/2024'

# datetime.strptime(texto, formato)
# O formato descreve COMO a string está escrita:
# '%d/%m/%Y' = dia/mês/ano
data_obj = datetime.strptime(data_texto, '%d/%m/%Y')

print(f'String original  : "{data_texto}"')
print(f'Objeto datetime  : {data_obj}')
print(f'Tipo             : {type(data_obj)}')
print()

# ── strftime: objeto data → texto formatado ───────────────────────────────
# Agora podemos formatar a data do jeito que quisermos
formatado = data_obj.strftime('%d de %B de %Y')  # 13 de August de 2024
print(f'Formatado        : {formatado}')

# Extraindo partes da data
print(f'Ano              : {data_obj.year}')
print(f'Mês              : {data_obj.month}')
print(f'Dia              : {data_obj.day}')

String original  : "13/08/2024"
Objeto datetime  : 2024-08-13 00:00:00
Tipo             : <class 'datetime.datetime'>

Formatado        : 13 de August de 2024
Ano              : 2024
Mês              : 8
Dia              : 13
